# Korean Response-Language Activation Steering

This notebook mirrors `examples/demo_actsteer_response_language.py` for the Korean response-language example. It runs plain English prompts through a baseline, a coefficient sweep, and several tuned examples so the steering effect is easy to inspect.

The steering vector was derived from Qwen2.5-3B hidden states using AutoSteer prompt-pair examples for `response_language=ko`.


### Installation

If running this from a new environment, set `RUN_INSTALL = True` in the cell below to install `vllm_hook_plugins` and the repository requirements. The default keeps installation disabled for an already-prepared local or cluster environment.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

SINGLE_GPU_DEVICE = os.environ.get("RESPONSE_LANGUAGE_CUDA_VISIBLE_DEVICES", "0")
visible = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()

if not visible:
    os.environ["CUDA_VISIBLE_DEVICES"] = SINGLE_GPU_DEVICE
    print(f"CUDA_VISIBLE_DEVICES was unset; using {SINGLE_GPU_DEVICE}.")
elif "," in visible:
    first_visible = visible.split(",", 1)[0].strip()
    os.environ["CUDA_VISIBLE_DEVICES"] = first_visible
    print(f"CUDA_VISIBLE_DEVICES had multiple GPUs ({visible}); using {first_visible}.")
else:
    print(f"CUDA_VISIBLE_DEVICES={visible}")

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "vllm_hook_plugins").exists():
    REPO_ROOT = CURRENT_DIR
elif CURRENT_DIR.name == "notebooks" and (CURRENT_DIR.parent / "vllm_hook_plugins").exists():
    REPO_ROOT = CURRENT_DIR.parent
else:
    candidates = [parent for parent in CURRENT_DIR.parents if (parent / "vllm_hook_plugins").exists()]
    if not candidates:
        raise RuntimeError(f"Could not locate vLLM-Hook repo root from {CURRENT_DIR}")
    REPO_ROOT = candidates[0]

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
RUN_INSTALL = False

print("Current dir :", CURRENT_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if RUN_INSTALL:
    if REQ_FILE.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)], check=True)
    else:
        print("Warning: requirement.txt not found at", REQ_FILE)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)], check=True)
else:
    print("Install step skipped. Set RUN_INSTALL=True if dependencies are not installed.")

plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)


CUDA_VISIBLE_DEVICES=0
Current dir : /u/yueling/actsteer_project/vLLM-Hook/notebooks
Repo root   : /u/yueling/actsteer_project/vLLM-Hook
Package dir : /u/yueling/actsteer_project/vLLM-Hook/vllm_hook_plugins
Req file    : /u/yueling/actsteer_project/vLLM-Hook/requirement.txt
Install step skipped. Set RUN_INSTALL=True if dependencies are not installed.


### Environment & Imports

Set multiprocessing and vLLM environment variables before importing vLLM, then verify that the notebook sees exactly one CUDA device.


In [2]:
import json
import gc
import multiprocessing as mp
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import torch
from vllm import SamplingParams
from vllm_hook_plugins import HookLLM

try:
    import langdetect
except Exception:
    langdetect = None

os.chdir(REPO_ROOT)
print("Working directory:", Path.cwd())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("CUDA available:", torch.cuda.is_available())

device_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("GPU count:", device_count)
if torch.cuda.is_available() and device_count > 0:
    for gpu_idx in range(device_count):
        print(f"GPU {gpu_idx} name:", torch.cuda.get_device_name(gpu_idx))

if device_count != 1:
    raise RuntimeError(
        "Expected exactly one visible CUDA GPU for this notebook. "
        f"Saw {device_count}. Restart the kernel after setting CUDA_VISIBLE_DEVICES "
        "to a single GPU or launch Jupyter from an LSF GPU allocation."
    )


/u/yueling/miniconda3/envs/vllm_hook_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /u/yueling/actsteer_project/vLLM-Hook
CUDA_VISIBLE_DEVICES: 0
CUDA available: True
GPU count: 1
GPU 0 name: NVIDIA A100-SXM4-80GB


### Configuration

Load the same Qwen Korean activation-steering config used by the Python example.


In [3]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"
LANGUAGE = "ko"
LANGUAGE_NAME = "Korean"
CONFIG_PATH = "model_configs/activation_steer/Qwen2.5-3B-Instruct-response-language-ko.json"
cache_dir = "./cache/"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

default_config = config["steering"]
vector_path = Path(default_config["vector_path"])
if not vector_path.is_absolute():
    vector_path = REPO_ROOT / vector_path

print(json.dumps(config, indent=2))
print("Config file    :", CONFIG_PATH)
print("Steering vector:", vector_path, "exists=", vector_path.exists())
assert vector_path.exists(), vector_path


{
  "steering": {
    "method": "add_vector",
    "coefficient": 80,
    "optimal_layer": 28,
    "vector_path": "steering_vectors/qwen25_response_language_ko.pt",
    "apply_at_all_positions": true,
    "apply_to_all_tokens": true
  }
}
Config file    : model_configs/activation_steer/Qwen2.5-3B-Instruct-response-language-ko.json
Steering vector: /u/yueling/actsteer_project/vLLM-Hook/steering_vectors/qwen25_response_language_ko.pt exists= True


### Initialize `HookLLM`

Create one hook-enabled vLLM instance and reuse it for every case, matching the Python example.


In [4]:
existing_llm = globals().get("llm")
if existing_llm is not None:
    print("Shutting down existing HookLLM before creating a new one.")
    try:
        existing_llm.llm_engine.engine_core.shutdown(timeout=0)
    except Exception as exc:
        print("Existing EngineCore shutdown skipped:", repr(exc))
    try:
        existing_llm.close()
    except Exception as exc:
        print("Existing HookLLM close skipped:", repr(exc))
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

llm = HookLLM(
    model=MODEL,
    worker_name="steer_hook_act",
    config_file=CONFIG_PATH,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    max_model_len=2048,
    trust_remote_code=True,
    dtype="auto",
    enforce_eager=True,
    enable_prefix_caching=True,
    enable_hook=True,
    tensor_parallel_size=1,
)


INFO 06-11 11:38:17 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': './cache/', 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'worker_extension_cls': 'vllm_hook_plugins.workers.steer_activation_worker.SteerHookActWorker', 'model': 'Qwen/Qwen2.5-3B-Instruct'}


WARNING 06-11 11:38:17 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1


INFO 06-11 11:38:19 [model.py:617] Resolved architecture: Qwen2ForCausalLM


INFO 06-11 11:38:19 [model.py:1752] Using max model len 2048


INFO 06-11 11:38:19 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:09<00:00,  4.67s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:09<00:00,  4.90s/it]
(EngineCore pid=1689139) 


(EngineCore pid=1689139) INFO 06-11 11:39:30 [default_loader.py:397] Loading weights took 9.88 seconds


(EngineCore pid=1689139) INFO 06-11 11:39:31 [gpu_model_runner.py:5132] Model loading took 5.79 GiB memory and 12.903201 seconds


(EngineCore pid=1689139) INFO 06-11 11:39:34 [gpu_worker.py:466] Available KV cache memory: 48.93 GiB
(EngineCore pid=1689139) INFO 06-11 11:39:34 [kv_cache_utils.py:1733] GPU KV cache size: 1,425,216 tokens
(EngineCore pid=1689139) INFO 06-11 11:39:34 [kv_cache_utils.py:1734] Maximum concurrency for 2,048 tokens per request: 695.91x
(EngineCore pid=1689139) INFO 06-11 11:39:34 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=1689139) INFO 06-11 11:39:34 [core.py:309] init engine (profile, create kv cache, warmup model) took 3.28 s


(EngineCore pid=1689139) INFO 06-11 11:39:36 [vllm.py:977] Asynchronous scheduling is enabled.
(EngineCore pid=1689139) WARNING 06-11 11:39:36 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=1689139) WARNING 06-11 11:39:36 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=1689139) INFO 06-11 11:39:36 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=1689139) INFO 06-11 11:39:36 [vllm.py:1234] Cudagraph is disabled under eager mode
(EngineCore pid=1689139) INFO 06-11 11:39:36 [compilation.py:312] Enabled custom fusions: norm_quant, act_quant


### Demo Helpers

These helpers mirror the Python example and add simple diagnostics for Hangul ratio, English-letter residue, and repeated-token oversteering.


In [5]:
PROMPTS = {
    "trip_planning": "Explain trip planning in simple terms.",
    "school_schedules": "Suggest first steps for improving school schedules.",
    "exercise_habits": "Write encouraging advice about exercise habits.",
    "trip_lesson": "Create a beginner-friendly lesson about trip planning.",
}

SCRIPT_RANGES = {
    "ko": [(0xAC00, 0xD7AF), (0x1100, 0x11FF), (0x3130, 0x318F)],
}


@dataclass(frozen=True)
class DemoCase:
    label: str
    prompt_key: str
    use_hook: bool
    method: Optional[str] = None
    coefficient: Optional[float] = None
    temperature: float = 0.0
    top_p: float = 1.0
    max_tokens: int = 120
    seed: Optional[int] = None


@dataclass(frozen=True)
class DemoResult:
    case: DemoCase
    text: str
    detected_language: Optional[str]
    target_script_ratio: float
    ascii_alpha_ratio: float
    char_repetition_ratio: float
    token_unique_ratio: float
    target_language_ok: bool


def detect_language(text: str) -> Optional[str]:
    if langdetect is None or not text.strip():
        return None
    try:
        return langdetect.detect(text)
    except Exception:
        return None


def script_char_count(text: str, language: str) -> int:
    ranges = SCRIPT_RANGES.get(language, [])
    count = 0
    for char in text:
        codepoint = ord(char)
        if any(start <= codepoint <= end for start, end in ranges):
            count += 1
    return count


def target_script_ratio(text: str, language: str) -> float:
    non_space = sum(1 for char in text if not char.isspace())
    if non_space == 0:
        return 0.0
    return script_char_count(text, language) / non_space


def ascii_alpha_ratio(text: str) -> float:
    non_space = sum(1 for char in text if not char.isspace())
    if non_space == 0:
        return 0.0
    ascii_alpha = sum(1 for char in text if "a" <= char.lower() <= "z")
    return ascii_alpha / non_space


def char_repetition_ratio(text: str) -> float:
    chars = [char for char in text if not char.isspace()]
    if not chars:
        return 0.0
    repeated = 0
    previous = None
    run = 0
    for char in chars:
        if char == previous:
            run += 1
        else:
            run = 1
            previous = char
        if run >= 4:
            repeated += 1
    return repeated / len(chars)


def response_tokens(text: str) -> list[str]:
    punctuation = ".,;:!?()[]{}<>\"'`*_-=+/\\|#~"
    return [token.strip(punctuation) for token in text.split() if token.strip(punctuation)]


def token_unique_ratio(text: str) -> float:
    tokens = response_tokens(text)
    if not tokens:
        return 0.0
    return len(set(tokens)) / len(tokens)


def response_language_ok(text: str, language: str) -> bool:
    return (
        target_script_ratio(text, language) >= 0.75
        and ascii_alpha_ratio(text) <= 0.03
        and char_repetition_ratio(text) <= 0.35
        and token_unique_ratio(text) >= 0.30
    )


def coefficient_label(case: DemoCase) -> str:
    if not case.use_hook:
        return "-"
    if case.method == "adjust_rs":
        return "n/a"
    if case.coefficient is None:
        return "json"
    return f"{case.coefficient:g}"


def apply_chat_template(llm: HookLLM, prompt: str) -> str:
    return llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
    )


def steering_override(default_config: dict, case: DemoCase) -> Optional[dict]:
    if not case.use_hook:
        return None
    steer = dict(default_config)
    if case.method is not None:
        steer["method"] = case.method
    if case.coefficient is not None:
        steer["coefficient"] = case.coefficient
    return steer


def sampling_params(llm: HookLLM, default_config: dict, case: DemoCase) -> SamplingParams:
    stop_token_ids = [
        token_id
        for token_id in (llm.tokenizer.eos_token_id, 32007)
        if token_id is not None
    ]
    kwargs = {
        "temperature": case.temperature,
        "top_p": case.top_p,
        "max_tokens": case.max_tokens,
        "stop_token_ids": stop_token_ids,
    }
    if case.seed is not None:
        kwargs["seed"] = case.seed
    steer = steering_override(default_config, case)
    if steer is not None:
        kwargs["extra_args"] = {"steer": steer}
    return SamplingParams(**kwargs)


def run_case(llm: HookLLM, default_config: dict, case: DemoCase) -> DemoResult:
    prompt = apply_chat_template(llm, PROMPTS[case.prompt_key])
    params = sampling_params(llm, default_config, case)
    output = llm.generate(prompt, params, use_hook=case.use_hook)[0]
    text = output.outputs[0].text.strip()
    llm.llm_engine.reset_prefix_cache()
    return DemoResult(
        case=case,
        text=text,
        detected_language=detect_language(text),
        target_script_ratio=target_script_ratio(text, LANGUAGE),
        ascii_alpha_ratio=ascii_alpha_ratio(text),
        char_repetition_ratio=char_repetition_ratio(text),
        token_unique_ratio=token_unique_ratio(text),
        target_language_ok=response_language_ok(text, LANGUAGE),
    )


def print_result(result: DemoResult) -> None:
    case = result.case
    hook = "on" if case.use_hook else "off"
    method = case.method or "none"
    print("=" * 88)
    print(
        f"[{case.label}] prompt={case.prompt_key} hook={hook} "
        f"method={method} coefficient={coefficient_label(case)} "
        f"temperature={case.temperature:g} top_p={case.top_p:g}"
    )
    print(
        f"Target={LANGUAGE_NAME} detected={result.detected_language} "
        f"ok={result.target_language_ok} "
        f"target_script_ratio={result.target_script_ratio:.3f} "
        f"ascii_alpha_ratio={result.ascii_alpha_ratio:.3f} "
        f"char_repetition_ratio={result.char_repetition_ratio:.3f} "
        f"token_unique_ratio={result.token_unique_ratio:.3f}"
    )
    print()
    print(result.text)


def print_summary(results: list[DemoResult]) -> None:
    print("=" * 88)
    print("Summary")
    print(
        f"{'Case':<28} {'Prompt':<18} {'Method':<10} {'Coeff':>6} "
        f"{'Detected':>9} {'OK':>4} {'Script':>7} {'ASCII':>7} "
        f"{'Repeat':>7} {'Unique':>7}"
    )
    for result in results:
        case = result.case
        method = case.method or "none"
        detected = result.detected_language or "?"
        print(
            f"{case.label:<28} {case.prompt_key:<18} {method:<10} "
            f"{coefficient_label(case):>6} {detected:>9} "
            f"{str(result.target_language_ok):>4} "
            f"{result.target_script_ratio:>7.3f} "
            f"{result.ascii_alpha_ratio:>7.3f} "
            f"{result.char_repetition_ratio:>7.3f} "
            f"{result.token_unique_ratio:>7.3f}"
        )


### Run the Demo

The coefficient sweep shows the transition from English output, to mixed output, to useful Korean output, and finally to oversteering.


In [6]:
cases = [
    DemoCase("baseline greedy", "trip_planning", use_hook=False),
    DemoCase("add_vector coeff 40", "trip_planning", use_hook=True, method="add_vector", coefficient=40),
    DemoCase("add_vector coeff 60", "trip_planning", use_hook=True, method="add_vector", coefficient=60),
    DemoCase("add_vector coeff 75", "trip_planning", use_hook=True, method="add_vector", coefficient=75),
    DemoCase("add_vector coeff 80", "trip_planning", use_hook=True, method="add_vector", coefficient=80),
    DemoCase("add_vector coeff 90", "trip_planning", use_hook=True, method="add_vector", coefficient=90),
    DemoCase("add_vector coeff 100", "trip_planning", use_hook=True, method="add_vector", coefficient=100),
    DemoCase("baseline school_schedules", "school_schedules", use_hook=False),
    DemoCase("tuned school_schedules", "school_schedules", use_hook=True, method="add_vector", coefficient=80),
    DemoCase("baseline exercise_habits", "exercise_habits", use_hook=False),
    DemoCase("tuned exercise_habits", "exercise_habits", use_hook=True, method="add_vector", coefficient=80),
    DemoCase("baseline trip_lesson", "trip_lesson", use_hook=False),
    DemoCase("tuned trip_lesson", "trip_lesson", use_hook=True, method="add_vector", coefficient=80),
]

results = []
try:
    print(f"Running {len(cases)} cases with one HookLLM instance for {MODEL}.")
    print(f"Target instruction: response_language={LANGUAGE_NAME} ({LANGUAGE})")
    for case in cases:
        result = run_case(llm, default_config, case)
        results.append(result)
        print_result(result)

    print_summary(results)

    by_label = {result.case.label: result for result in results}
    assert not by_label["baseline greedy"].target_language_ok
    assert not by_label["add_vector coeff 40"].target_language_ok
    assert not by_label["add_vector coeff 60"].target_language_ok
    assert by_label["add_vector coeff 75"].target_language_ok
    assert by_label["add_vector coeff 80"].target_language_ok
    assert not by_label["add_vector coeff 90"].target_language_ok
    assert not by_label["add_vector coeff 100"].target_language_ok
    assert by_label["tuned school_schedules"].target_language_ok
    assert by_label["tuned exercise_habits"].target_language_ok
    assert by_label["tuned trip_lesson"].target_language_ok
    print("Output sanity checks passed.")
finally:
    llm.close()
    print("HookLLM closed.")


Running 13 cases with one HookLLM instance for Qwen/Qwen2.5-3B-Instruct.
Target instruction: response_language=Korean (ko)


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00,  5.52it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00,  5.47it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.78s/it, est. speed input: 13.65 toks/s, output: 43.12 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.78s/it, est. speed input: 13.65 toks/s, output: 43.12 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.78s/it, est. speed input: 13.65 toks/s, output: 43.12 toks/s]

(EngineCore pid=1689139) INFO 06-11 11:40:12 [block_pool.py:482] Successfully reset prefix cache
[tuned trip_lesson] prompt=trip_lesson hook=on method=add_vector coefficient=80 temperature=0 top_p=1
Target=Korean detected=ko ok=True target_script_ratio=0.944 ascii_alpha_ratio=0.000 char_repetition_ratio=0.000 token_unique_ratio=0.765

여기 초보자용 여행 계획법 수업을 만들어 드리겠습니다. 여행을 가려는 때가 되면, 일정을 어떻게 짜야 할지 모르는 경우가 많을 테니까요. 이 수업은 여행 가는 날짜부터 끝까지 주요 단계를 설명할 예정입니다. 

1. 여행 일정 계획 시작
   여행 일정 계획은 여행 시작 전에 시작해야 합니다. 여행 일정 계획은 여행 일정을 세우는 것부터 시작되며, 여행
Summary
Case                         Prompt             Method      Coeff  Detected   OK  Script   ASCII  Repeat  Unique
baseline greedy              trip_planning      none            -        en False   0.000   0.941   0.000   0.710
add_vector coeff 40          trip_planning      add_vector     40        en False   0.000   0.920   0.000   0.684
add_vector coeff 60          trip_planning      add_vector     60        ko False   0.678   0.256   0.000   0.948
a